In [1]:
import pandas as pd
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats

Importing data

In [ ]:
matrix_exp = pd.read_csv("../results/heatmaps/matrix_exp_duplex.tsv", sep="\t", index_col=0)
matrix_pmhc_score = pd.read_csv("../results/heatmaps/matrix_score_netmhcpan.tsv", sep="\t", index_col=0)
matrix_bl = pd.read_csv("../results/heatmaps/matrix_bl_raw.tsv", sep="\t", index_col=0)
matrix_bl_cond = pd.read_csv("../results/heatmaps/matrix_bl_cond.tsv", sep="\t", index_col=0)
matrix_bl_scaled= pd.read_csv("../results/heatmaps/matrix_bl_rescaled.tsv", sep="\t", index_col=0)
matrix_af = pd.read_csv("../results/heatmaps/matrix_af_interface.tsv", sep="\t", index_col=0)

# List of peptides to test 

df_pep = pd.read_csv("/home/projects2/morni2/randompeps/randompeps_9mer_1000000.pep", header=None, comment="#", names=["peptide"])

cleaned_df_pep = df_pep.dropna(subset=["peptide"]).reset_index(drop=True)
# df_all = cleaned_df_pep.copy()

In [12]:
print(matrix_exp.head())


NameError: name 'matrix_exp' is not defined

### Hybrid Score Design: pMHC $\times$ BLOSUM62

To rank mutant peptide candidates against experimental binding data, we combine NetMHCpan presentation predictions with BLOSUM62 sequence conservation using the following naive formulation:

$$S_{i, j} = \text{pMHC}_{i, j} \times \left(1 + \text{BL}_{\text{scaled}, i, j}\right)$$

$$S_{i, j} = \text{BL}_{i, j} \times \left(1 - \text{pMHC}_{i, j}\right) + \text{pMHC}_{i, j}$$

#### Key Rationales:
* **MHC Presentation as a Mandatory Gate ($\text{pMHC} \to 0$):**  
  Peptide presentation by the MHC is an absolute biological prerequisite. If a mutation destroys binding at anchor positions (P2, P9), $\text{pMHC} \to 0$, forcing $S \to 0$ regardless of favorable physicochemical properties elsewhere.
* **Positive Modulation on Permissive Positions ($\text{pMHC} \approx 1$):**  
  On non-anchor positions where MHC binding is widely tolerated, $\text{pMHC}$ acts as a neutral baseline ($1.0$), allowing the $(1 + \text{BL}_{\text{scaled}})$ factor to discriminate mutants based on sequence conservation and TCR compatibility.
* **Column-wise Min-Max Scaling ($\text{BL}_{\text{scaled}} \in [0, 1]$):**  
  Standard global scaling on BLOSUM62 introduces an artifact: Tryptophan's extreme self-identity score ($+11$) artificially suppresses non-W positions below $0.5$. Scaling each column independently relative to its position's WT residue ensures that:
  1. Every wild-type residue anchors at exactly $1.0$.
  2. Scores remain strictly positive ($[1.0, 2.0]$ multiplier), preventing rank inversions caused by negative raw BLOSUM values.

In [ ]:
scoring_mat = matrix_pmhc_score * (1 + matrix_bl_scaled)
# print(matrix_pmhc_score.head())
# print(matrix_bl_scaled.head())
print(scoring_mat.head())

                    1         2         3         4         5         6  \
Peptide_res                                                               
A            1.003391  1.284184  1.046129  1.075355  0.294708  1.218871   
C            0.322615  0.635840  0.444676  0.392694  0.023704  0.660907   
D            1.386900  0.327703  0.814468  0.988959  0.256493  0.720380   
E            1.295710  0.486721  0.783880  1.253564  0.431582  0.885918   
F            0.992410  1.062150  1.825030  0.742672  0.240422  1.167659   

                    7         8         9  
Peptide_res                                
A            1.013440  1.253273  0.541052  
C            0.932194  0.666476  0.240156  
D            0.883406  1.197352  0.006025  
E            0.994032  1.825030  0.059099  
F            1.227409  0.727278  1.098669  
                    1         2         3         4         5         6  \
Peptide_res                                                               
A            0.1

In [14]:
def score_full_peptides(input_df, scoring_matrix, pep_col,  score_prefix, aggregation="sum"):
    """
    The input must be a dataframe with at least a column containing the peptides being tested. 
    It is then returned with 2 additionnal columns containing the scored peptides and their rank.
    """

    df = input_df.copy()
    
    sc_mat = scoring_matrix.copy()
    sc_mat.columns = sc_mat.columns.astype(int)

    scores = []
    
    for pep in df[pep_col]:
        if len(pep) != 9:
            scores.append(np.nan)
            continue
            
        pred_pos_scores = []
        
        for pos, aa in enumerate(pep, start=1):
            if aa in sc_mat.index and pos in sc_mat.columns:
                pred_pos_scores.append(sc_mat.loc[aa, pos])
            else:
                pred_pos_scores.append(np.nan)
                
        if aggregation == "sum":
            final_score = np.sum(pred_pos_scores) if not any(np.isnan(pred_pos_scores)) else np.nan
        elif aggregation == "prod":
            final_score = np.prod(pred_pos_scores) if not any(np.isnan(pred_pos_scores)) else np.nan
            
        scores.append(final_score)
        
    df[f"{score_prefix}_score"] = scores
    df[f"{score_prefix}_rank"] = df[f"{score_prefix}_score"].rank(ascending=False, method="min")
        
    return df

In [9]:
df_all = score_full_peptides(df_all, matrix_exp, pep_col="peptide",score_prefix="exp", aggregation="sum")
df_all = score_full_peptides(df_all, scoring_mat, pep_col="peptide", score_prefix="pred", aggregation="sum")

top_hits_pred = df_all.sort_values(by="pred_score", ascending=False)
print(top_hits_pred.head(10))

top_hits_exp = df_all.sort_values(by="exp_score", ascending=False)
print(top_hits_exp.head(10))

          peptide  exp_score  exp_rank  pred_score  pred_rank
650853  NLFKRMRKL  15.988712     522.0   14.245247        1.0
748464  NIERRIEEL  21.080991      35.0   14.239423        2.0
819444  NLYKRMASL  12.332569    3215.0   14.087433        3.0
193659  NILRRLYNL  18.081211     165.0   14.052776        4.0
11157   QLMRRLAEL  23.353857       8.0   14.026697        5.0
728973  TVYRRLKEL  18.716694     129.0   14.007224        6.0
366138  ELQERVYEL  18.677872     133.0   13.959839        7.0
477951  ALYRRITTL  16.202841     465.0   13.956987        8.0
908502  NLFKRKPKL  12.819129    2591.0   13.941467        9.0
764956  RVYRRVFHL  17.252170     254.0   13.882271       10.0
          peptide  exp_score  exp_rank  pred_score  pred_rank
569833  YLMKRGFEI  26.259803       1.0   12.734704      341.0
581957  FLDNRIFDI  24.771325       2.0   12.007876     2236.0
282568  AAFRRYFDI  24.337944       3.0   12.535982      604.0
955692  SPFRRVEDV  24.041959       4.0   12.954938      173.0
507579  

In [ ]:
# def score_cross_metrics(df_input, exp_prefix, score_prefix, return_dict=False):
#     """
#     Compute correlation scores and look for matching hits between an experimental 
#     and predictive score (or 2 scores, the first being the reference one)
#     """

#     clean_df = df_input.dropna(subset=[f"{score_prefix}_score", f"{exp_prefix}_score"])
    

#     scc, pval = stats.spearmanr(clean_df[f"{score_prefix}_score"], clean_df[f"{exp_prefix}_score"])
#     pcc, pval2 = stats.pearsonr(clean_df[f"{score_prefix}_score"], clean_df[f"{exp_prefix}_score"])


#     top_k_exp = set(clean_df.nlargest(100, f"{exp_prefix}_score")["peptide"])
#     top_k_pred = set(clean_df.nlargest(100, f"{score_prefix}_score")["peptide"])

#     overlap_100 = len(top_k_exp.intersection(top_k_pred))

#     # Checking the predicted Top 1000 (first 0.1 %)
#     top_1000_pred = set(clean_df.nlargest(1000, f"{score_prefix}_score")["peptide"])
#     overlap_in_top1000 = len(top_k_exp.intersection(top_1000_pred))

#     print(f"[{score_prefix} vs {exp_prefix}]")
#     print(f"  SCC: {scc:.4f} (p-val: {pval:.2e})")
#     print(f"  PCC: {pcc:.4f} (p-val: {pval2:.2e})")
#     print(f"  Overlap in top 100: {overlap_100}/100 peptides")
#     print(f"  Top 100 captured in predicted top 1000: {overlap_in_top1000}/100 peptides\n")


#     if return_dict:
#         return {
#             "model": score_prefix,
#             "exp_ref": exp_prefix,
#             "scc": scc,
#             "pcc": pcc,
#             "top100_overlap": overlap_100,
#             "top100_in_top1000": overlap_in_top1000
#         }

# version with percentiles instead of absolute numbers of top hits

def score_cross_metrics(df_input, exp_prefix, score_prefix, return_dict=True, percentiles=[0.1, 1]):
    """
    Compute correlation scores and look for matching hits between an experimental 
    and predictive score (or 2 scores, the first being the reference one)
    """

    clean_df = df_input.dropna(subset=[f"{score_prefix}_score", f"{exp_prefix}_score"])
    n_tot = len(clean_df)
    

    scc, pval = stats.spearmanr(clean_df[f"{score_prefix}_score"], clean_df[f"{exp_prefix}_score"])
    pcc, pval2 = stats.pearsonr(clean_df[f"{score_prefix}_score"], clean_df[f"{exp_prefix}_score"])

    n_ref = max(1, int(n_tot * 0.001))
    top_ref_exp = set(clean_df.nlargest(n_ref, f"{exp_prefix}_score")["peptide"])

    res = {
        "model": score_prefix,
        "scc": round(float(scc), 4),
        "pcc": round(float(pcc), 4),
        "N_ref_0.1%": n_ref
    }
    
    print(f"[{score_prefix} vs {exp_prefix}]")
    print(f"  SCC: {scc:.4f} (p-val: {pval:.2e})")
    print(f"  PCC: {pcc:.4f} (p-val: {pval2:.2e})")

    for p in percentiles:
        k_pred = max(1, int(n_tot * (p / 100.0)))
        top_k_pred = set(clean_df.nlargest(k_pred, f"{score_prefix}_score")["peptide"])
        overlap = len(top_ref_exp.intersection(top_k_pred))
        recall = overlap / n_ref * 100.0
        res[f"recall_top{p}%"] = round(recall, 2)
        res[f"overlap_top{p}%"] = overlap
        print(f"Top 0.1% captured in predicted top {p}%: {recall:.1f}% ({overlap}/{n_ref} peptides)")

    if return_dict:
        return res

Now trying with the netMHCpan score alone to see if the BLOSUM addition was worth it!

In [ ]:
df_all = score_full_peptides(df_all, matrix_pmhc_score, pep_col="peptide", score_prefix="pMHC", aggregation="sum")

[pMHC vs exp]
  SCC: 0.5729 (p-val: 0.00e+00)
  PCC: 0.5846 (p-val: 0.00e+00)
  Overlap in top 100: 0/100 peptides
  Top 100 captured in predicted top 1000: 10/100 peptides



Conversely, trying to score with only the BLOSUM matrix

In [27]:
df_all = score_full_peptides(df_all, matrix_bl_scaled, pep_col="peptide", score_prefix="BL", aggregation="sum")

And with a 2nd version of the naive score, where we just multiply blosum and netMHCpan scores:

In [ ]:
scoring_mat2 = matrix_pmhc_score * matrix_bl_scaled
df_all = score_full_peptides(df_all, scoring_mat2, pep_col="peptide", score_prefix="product", aggregation="sum")

NameError: name 'benchmark' is not defined

## Updated score design with AF interface score

In [ ]:
full_scoring_mat = matrix_pmhc_score * matrix_bl_scaled * matrix_af
df_all = score_full_peptides(df_all, full_scoring_mat, pep_col="peptide", score_prefix="full", aggregation="sum")

[full vs exp]
  SCC: 0.7333 (p-val: 0.00e+00)
  PCC: 0.7499 (p-val: 0.00e+00)
  Overlap in top 100: 23/100 peptides
  Top 100 captured in predicted top 1000: 53/100 peptides



First, trying to score with the AF output only

In [28]:
df_all = score_full_peptides(df_all, matrix_af, pep_col="peptide", score_prefix="AF3", aggregation="sum")

Alternative score without BLOSUM to see how this impacts the global results

In [20]:
partial_scoring_mat_woBL = matrix_pmhc_score * matrix_af
df_all = score_full_peptides(df_all, partial_scoring_mat_woBL, pep_col="peptide", score_prefix="part_woBL", aggregation="sum")

Alternative score without pMHC score to see how this impacts the global results

In [22]:
partial_scoring_mat_woMHC = matrix_bl_scaled * matrix_af
df_all = score_full_peptides(df_all, partial_scoring_mat_woMHC, pep_col="peptide", score_prefix="part_woMHC", aggregation="sum")

Running all benchmark metrics at once and summarizing results in a df

In [6]:
benchmark = [
    score_cross_metrics(df_all, "exp", model) 
    for model in ["pMHC", "BL", "AF3", "pred", "product", "full", "part_woMHC", "part_woBL"]
]
df_summary = pd.DataFrame(benchmark).set_index("model")

[pMHC vs exp]
  SCC: 0.5729 (p-val: 0.00e+00)
  PCC: 0.5846 (p-val: 0.00e+00)
Top 0.1% captured in predicted top 0.1%: 4.6% (41/889 peptides)
Top 0.1% captured in predicted top 1%: 38.0% (338/889 peptides)
[BL vs exp]
  SCC: 0.6775 (p-val: 0.00e+00)
  PCC: 0.6971 (p-val: 0.00e+00)
Top 0.1% captured in predicted top 0.1%: 25.0% (222/889 peptides)
Top 0.1% captured in predicted top 1%: 67.3% (598/889 peptides)
[AF3 vs exp]
  SCC: 0.3818 (p-val: 0.00e+00)
  PCC: 0.4072 (p-val: 0.00e+00)
Top 0.1% captured in predicted top 0.1%: 6.6% (59/889 peptides)
Top 0.1% captured in predicted top 1%: 27.9% (248/889 peptides)
[pred vs exp]
  SCC: 0.7229 (p-val: 0.00e+00)
  PCC: 0.7365 (p-val: 0.00e+00)
Top 0.1% captured in predicted top 0.1%: 24.1% (214/889 peptides)
Top 0.1% captured in predicted top 1%: 72.0% (640/889 peptides)
[product vs exp]
  SCC: 0.7064 (p-val: 0.00e+00)
  PCC: 0.7243 (p-val: 0.00e+00)
Top 0.1% captured in predicted top 0.1%: 25.0% (222/889 peptides)
Top 0.1% captured in predict

## Export data

In [ ]:
# df_all.to_csv("../results/peptides_scored_summary.csv", index=False)
# df_summary.to_csv("../results/peptides_scored_metrics_perc.csv", index="model")

In [3]:
df_summary 

,scc,pcc,N_ref_0.1%,recall_top0.1%,overlap_top0.1%,recall_top1%,overlap_top1%
model,,,,,,,
pMHC,0.5729,0.5846,889,4.61,41,38.02,338
BL,0.6775,0.6971,889,24.97,222,67.27,598
AF3,0.3818,0.4072,889,6.64,59,27.90,248
pred,0.7229,0.7365,889,24.07,214,71.99,640
product,0.7064,0.7243,889,24.97,222,69.29,616
full,0.7333,0.7499,889,26.88,239,72.78,647
part_woMHC,0.7069,0.7257,889,27.67,246,72.55,645
part_woBL,0.6772,0.6930,889,21.03,187,65.58,583


We can also import the latest results dataframe to work with it without having to compute again all scores

In [2]:
# df_all = pd.read_csv("../results/peptides_scored_summary.csv")
df_summary = pd.read_csv("../results/peptides_scored_metrics_perc.csv", index_col="model")

Exporting top 0.1% hits of few scores to compare to experimental data trough logos

In [11]:
n_ref = int(len(df_all.dropna()) * 0.001)

# Exporting top hits for these models
models_to_plot = ["pMHC", "AF3"]

for m in models_to_plot:
    col = f"{m}_score"
    top_peptides = df_all.nlargest(n_ref, col)["peptide"]
    
    # Just exporting with one peptide a line
    out_file = f"../data/scoring/logo_top0.1pct_{m}.txt"
    top_peptides.to_csv(out_file, index=False, header=False)